# G1 Academy Bonus - Task 6: basic locomotion with the wrapper

## Introduction
You drive the robot through the finished `sdk_wrapper.G1` wrapper -- no native `LocoClient`, no DDS subscriber classes. `g1.loco_move(vx, vy, vyaw, duration_s)` is the one call this whole task is about: a timed velocity command that auto-stops for you. You will read odometry, drive a few timed segments, and assemble them into a simple multi-stop "U" path that Day 3 reuses (adding an arm gesture at each stop).

**Using Codex/AI for this task:** `loco_move`/`loco_stop`/`get_odom` are finished, documented methods on `sdk_wrapper.G1`. If a cell is not obvious, paste the signature (or the matching line from `wrapper_cheatsheet.html`) into Codex, then read what it produced before you run it against the robot.

In [ ]:
import sys
import time
sys.path.append("..")
from sdk_wrapper import G1

g1 = G1(iface="eth0", domain_id=0)

# Always step up through the modes before moving.
g1.damp_mode()
g1.prepare_mode()
g1.walk_mode()

## Task 1 - Read odometry: `get_odom()`
`g1.get_odom()` is your ground truth for how far the robot actually went -- open-loop timing drifts, so read it before and after each move rather than trusting the commanded velocity × duration.

In [ ]:
g1.get_odom()

## Task 2 - `g1.loco_move(vx, vy, vyaw, duration_s)`
- `loco_move(vx, vy, vyaw, duration_s=2)` drives for exactly `duration_s` seconds, then auto-stops (calls `loco_stop()` for you) -- the common case.
- `loco_move(vx, vy, vyaw, duration_s=None)` fires one `Move()` RPC and returns immediately, no auto-stop -- only use this when you manage the stop yourself.
- `loco_stop()` stops immediately -- call any time you need to bail out of a move early.

`vx`/`vy` are m/s (forward/strafe), `vyaw` is rad/s. Start small (≈0.15 m/s) with a spotter in a clear space.

In [ ]:
g1.loco_move(vx=0.2, vy=0, vyaw=0, duration_s=2)

## Task 3 - a timed multi-stop path ("U" path)
A path is just a few `loco_move(...)` calls back to back: **forward, turn, forward, turn, forward** traces an approximate "U" in three straight legs with a turn between them, then a final `loco_stop()`.

Hints for `walk_u_path()`:
- A forward leg is `g1.loco_move(vx=0.2, vy=0, vyaw=0, duration_s=5)`.
- A turn is `g1.loco_move(vx=0, vy=0, vyaw=vel_rot, duration_s=5)`, where `vel_rot` is the yaw rate (rad/s) that completes a quarter turn (`angle_rot = 3.14/2` rad) in that time: `vel_rot = angle_rot / 5`.
- Do three forward legs with two turns between them, then call `g1.loco_stop()`.

In [ ]:
angle_rot = 3.14 / 2      # a quarter turn, in radians
vel_rot = angle_rot / 5   # yaw rate that completes the turn in 5 seconds

def walk_u_path():
    g1.loco_move(vx=0.2, vy=0, vyaw=0, duration_s=5)       # forward
    g1.loco_move(vx=0, vy=0, vyaw=vel_rot, duration_s=5)   # turn
    g1.loco_move(vx=0.2, vy=0, vyaw=0, duration_s=5)       # forward
    g1.loco_move(vx=0, vy=0, vyaw=vel_rot, duration_s=5)   # turn
    g1.loco_move(vx=0.2, vy=0, vyaw=0, duration_s=5)       # forward
    g1.loco_stop()

walk_u_path()

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.